# 07a · 이토 적분과 등거리성 (The Itô integral and the Itô isometry)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Øksendal §3.1 (Itô 적분의 구성), §3.2 (등거리성·마팅게일), §3.3 (확장) — 주 교재 · Shreve II §4.2 (단순 피적분함수), §4.3 (일반 피적분함수) — 그림과 직관 · Lawler 2e §9.1 (브라운 운동에 대한 적분) — 간결한 증명 · G&S 4e §13.7–13.8 (stochastic calculus, the Itô integral; 절 번호는 판에 따라 다를 수 있으니 목차 확인) |
| 선수 노트북 | 06b (이차변동 $\sum\Delta B_i^2\to T$), 06d (BM 마팅게일·선택적 정지 정리), 06e (GBM·브라운 다리·드리프트 BM) |
| 예상 소요 | 100분 |
| 상태 | draft |

06b에서 브라운 운동(Brownian motion)의 이차변동(quadratic variation)이 0이 아니라 $T$ 로 남는 것을 보았다. 이 노트북은 그 사실의 첫 번째 결과를 다룬다: **브라운 운동에 대한 리만합은 표본점을 어디에 찍느냐에 따라 극한이 달라진다.** 왼쪽 끝점을 택한 것이 Itô 적분이고, 그 선택 덕분에 적분은 평균 0인 마팅게일이 되며 분산은 등거리성(isometry) $E[(\int H\,dB)^2]=E\int H^2dt$ 로 계산된다. 05b의 마팅게일 변환 $\sum H_{n-1}\Delta M_n$ 이 연속시간으로 옮겨온 것이다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy import stats
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.brownian import brownian_paths, coarsen
from spkit.plots import plot_hist_vs_pdf

SEED, rng = rng_for("07a")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_ISO = scale(40_000)     # 등거리성 표용 별도 경로 집합
N_FINE = 1000             # 세밀 격자 (스케일하지 않음: 중점이 필요해 n=500 의 두 배)
T = 1.0
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_ISO={N_ISO}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (06e) 기하 브라운 운동 $S_t=S_0\exp((\mu-\sigma^2/2)t+\sigma B_t)$ 의 기댓값 $E[S_t]$ 는? 지수 안의 $-\sigma^2/2$ 는 무엇을 상쇄하는가?

<details><summary>정답</summary>

$E[S_t]=S_0e^{\mu t}$. $E[e^{\sigma B_t}]=e^{\sigma^2t/2}$ 이므로 $-\sigma^2t/2$ 가 이를 정확히 상쇄한다. (07b 의 Itô 공식에서 보듯 $-\sigma^2/2$ 는 Itô 보정항이다.)

</details>

**Q2.** (06e) 브라운 다리 $X_t=B_t-\frac tTB_T$ 의 공분산 $\mathrm{Cov}(X_s,X_t)$ ($s\le t$)는?

<details><summary>정답</summary>

$s-\frac{st}T=\frac{s(T-t)}T$. 특히 $\mathrm{Var}(X_t)=t(T-t)/T$ 로 양 끝에서 0.

</details>

**Q3.** (06e) 드리프트 브라운 운동 $X_t=\mu t+\sigma B_t$ ($\mu<0$) 가 수준 $a>0$ 에 언젠가 도달할 확률은?

<details><summary>정답</summary>

$P(\sup_tX_t\ge a)=e^{2\mu a/\sigma^2}=e^{-2|\mu|a/\sigma^2}<1$. 지수 마팅게일 $e^{\theta X_t}$ 에서 $\theta=-2\mu/\sigma^2$ 로 잡아 선택적 정지 정리 (optional stopping) 를 쓴다.

</details>

**Q4.** (06d) 0에서 출발한 BM이 $[-a,b]$ 를 탈출할 때 $b$ 에 먼저 닿을 확률과 기대 탈출 시간은? 어떤 마팅게일을 썼나?

<details><summary>정답</summary>

$P(\text{hit }b\text{ first})=a/(a+b)$ ($B_t$ 마팅게일), $E[\tau]=ab$ ($B_t^2-t$ 마팅게일). 오늘 등장하는 $(B_T^2-T)/2$ 는 바로 이 두 번째 마팅게일의 절반이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **분할과 리만합.** $0=t_0<t_1<\dots<t_n=T$, $\Delta t=T/n$, $\Delta B_i=B_{t_{i+1}}-B_{t_i}$. 피적분함수(integrand) $H$ 에 대해 세 가지 리만합(Riemann sum)을 구별한다:
  왼쪽 끝점 합 $L_n=\sum_iH_{t_i}\Delta B_i$, 중점 합 $M_n=\sum_iH_{(t_i+t_{i+1})/2}\Delta B_i$, 오른쪽 끝점 합 $R_n=\sum_iH_{t_{i+1}}\Delta B_i$.
- **적합(adapted) 단순 과정.** $H_t=\sum_i\xi_i\mathbf 1_{[t_i,t_{i+1})}(t)$ 에서 각 $\xi_i$ 가 $\mathcal F_{t_i}$-가측(시각 $t_i$ 까지의 정보만 사용)이고 $E[\xi_i^2]<\infty$.
- **단순 과정의 Itô 적분.** $\int_0^TH\,dB:=\sum_i\xi_i\Delta B_i$ — 판돈 $\xi_i$ 는 구간의 **왼쪽** 끝에서 정해지고, 그 뒤에 오는 증분 $\Delta B_i$ 를 곱한다.
- **일반 적합 $H$ 의 Itô 적분.** $E\int_0^TH_t^2dt<\infty$ 이면 단순 과정 $H^{(n)}\to H$ ($L^2(dt\times dP)$) 를 잡고 $\int H\,dB:=L^2\text{-}\lim\int H^{(n)}dB$. 아래 등거리성이 $\{\int H^{(n)}dB\}$ 가 코시열임을 보장하므로 극한이 존재하고, 근사열 선택에 무관하다 (Øksendal §3.1).
- **Itô vs Stratonovich.** 같은 피적분함수라도 왼쪽 끝점(Itô)과 중점(Stratonovich, $\circ\,dB$)의 극한이 다르다: $\int_0^TB\,dB=(B_T^2-T)/2$, $\int_0^TB\circ dB=B_T^2/2$.
- **마팅게일 성질.** $M_t=\int_0^tH\,dB$ 는 $\mathcal F_t$-마팅게일: $E[M_T\mid\mathcal F_s]=M_s$, 특히 $E[M_T]=0$, $E[M_s(M_T-M_s)]=0$ (직교 증분).

### 2.2 정리 1 — Itô 등거리성 (단순 적합 과정; Øksendal §3.1, Shreve II §4.2)

$H_t=\sum_i\xi_i\mathbf 1_{[t_i,t_{i+1})}$ 가 적합 단순 과정이면

$$E\Big[\int_0^TH\,dB\Big]=0,\qquad E\Big[\Big(\int_0^TH\,dB\Big)^2\Big]=E\int_0^TH_t^2\,dt=\sum_iE[\xi_i^2]\,\Delta t_i .$$

가정이 왜 필요한가:
- **적합성**: $\xi_i$ 가 $\mathcal F_{t_i}$-가측이어야 $\Delta B_i$ 와 독립이 되어 $E[\xi_i\Delta B_i]=E[\xi_i]E[\Delta B_i]=0$, $E[\xi_i^2\Delta B_i^2]=E[\xi_i^2]\Delta t_i$ 가 성립한다. 오른쪽 끝점($\xi_i=B_{t_{i+1}}$)은 이를 깨뜨린다 (E3).
- **제곱 적분가능** $E[\xi_i^2]<\infty$: 우변이 유한하고 교차항의 기댓값을 계산할 수 있어야 한다.
- **증분의 독립성과 $E[\Delta B_i^2]=\Delta t_i$**: 브라운 운동의 정의(06a)가 쓰이는 유일한 곳. 교차항 $i<j$ 에서 $\Delta B_j$ 가 $\xi_i\xi_j\Delta B_i$ 와 독립이라 사라진다.

<details><summary>증명</summary>

$I=\sum_i\xi_i\Delta B_i$ 라 하자.

**(평균)** 각 $i$ 에서 $\xi_i$ 는 $\mathcal F_{t_i}$-가측이고 $\Delta B_i$ 는 $\mathcal F_{t_i}$ 와 독립이므로 $E[\xi_i\Delta B_i]=E[\xi_i]\,E[\Delta B_i]=0$. 더하면 $E[I]=0$.

**(제곱)** 전개하면
$$I^2=\sum_i\xi_i^2\Delta B_i^2+2\sum_{i<j}\xi_i\xi_j\Delta B_i\Delta B_j .$$
- 대각항: $\xi_i^2\perp\Delta B_i$ 이므로 $E[\xi_i^2\Delta B_i^2]=E[\xi_i^2]\,E[\Delta B_i^2]=E[\xi_i^2]\Delta t_i$.
- 교차항 $i<j$: $\xi_i\xi_j\Delta B_i$ 는 $\mathcal F_{t_j}$-가측이고($t_{i+1}\le t_j$), $\Delta B_j$ 는 $\mathcal F_{t_j}$ 와 독립이며 $E[\Delta B_j]=0$. 따라서 $E[\xi_i\xi_j\Delta B_i\Delta B_j]=E[\xi_i\xi_j\Delta B_i]\,E[\Delta B_j]=0$.

합하면 $E[I^2]=\sum_iE[\xi_i^2]\Delta t_i=E\int_0^TH_t^2dt$ (단순 과정이므로 적분은 계단 함수의 합). ∎

일반 $H$ 로의 확장: $\|\int H^{(n)}dB-\int H^{(m)}dB\|_{L^2(P)}=\|H^{(n)}-H^{(m)}\|_{L^2(dt\times dP)}$ 이므로 근사열이 코시열이면 적분열도 코시열이고, 등거리성은 $L^2$ 극한에서 보존된다 (Øksendal §3.1 Lemma 3.1.5 부근, §3.2).

</details>

### 2.3 정리 2 — $\int_0^TB\,dB=(B_T^2-T)/2$ 와 세 가지 리만합의 극한 (Shreve II §4.2, Lawler 2e §9.1)

$H=B$ 에 대해 임의의 $n$ 에서 대수 항등식

$$L_n=\tfrac12\Big(B_T^2-\sum_i\Delta B_i^2\Big),\qquad R_n=\tfrac12\Big(B_T^2+\sum_i\Delta B_i^2\Big)$$

이 성립하고, 이차변동 $QV_n:=\sum_i\Delta B_i^2\to T$ ($L^2$, 06b) 이므로

$$L_n\to\frac{B_T^2-T}2,\qquad M_n\to\frac{B_T^2}2,\qquad R_n\to\frac{B_T^2+T}2 .$$

또한 오차의 평균제곱은 정확히 $E[(L_n-\tfrac{B_T^2-T}2)^2]=\tfrac{T^2}{2n}$, $E[(M_n-\tfrac{B_T^2}2)^2]=\tfrac{T^2}{4n}$ 이다.

가정이 왜 필요한가:
- **이차변동이 $T$ 로 수렴** (06b): 세 극한을 갈라놓는 유일한 이유. 유계변동 적분자였다면 $QV_n\to0$ 이라 셋이 같았을 것이다.
- **격자를 세분화해도 같은 경로를 쓴다** (`coarsen`): 극한을 '경로별'로 말하려면 같은 $\omega$ 를 유지해야 한다. 4절에서 해상도마다 새 경로를 뽑지 않는 이유다.

<details><summary>증명</summary>

**(왼쪽·오른쪽)** $B_{t_i}\Delta B_i=\tfrac12\big[(B_{t_i}+\Delta B_i)^2-B_{t_i}^2-\Delta B_i^2\big]=\tfrac12\big[B_{t_{i+1}}^2-B_{t_i}^2-\Delta B_i^2\big]$ 를 $i$ 에 대해 더하면 앞 두 항은 망원합(telescoping) 이 되어 $L_n=\tfrac12(B_T^2-QV_n)$. 한편 $R_n-L_n=\sum_i(B_{t_{i+1}}-B_{t_i})\Delta B_i=QV_n$ 이므로 $R_n=\tfrac12(B_T^2+QV_n)$.

오차는 $L_n-\tfrac{B_T^2-T}2=\tfrac12(T-QV_n)$. $\Delta B_i^2$ 는 평균 $\Delta t$, 분산 $2\Delta t^2$ 인 i.i.d. 이므로 $E[QV_n]=T$, $\mathrm{Var}(QV_n)=n\cdot2\Delta t^2=2T^2/n$ (06b). 따라서 $E[(L_n-\tfrac{B_T^2-T}2)^2]=\tfrac14\mathrm{Var}(QV_n)=\tfrac{T^2}{2n}\to0$: $L^2$ 수렴이고 RMS 는 $T/\sqrt{2n}$. $R_n$ 은 $L_n+QV_n$ 이므로 같은 속도로 $(B_T^2+T)/2$ 로 간다.

**(중점)** 각 구간을 반으로 나눠 $\Delta B_i=\delta_i^{(1)}+\delta_i^{(2)}$ ($\delta^{(1)}$ 은 앞 반, $\delta^{(2)}$ 는 뒤 반, 각각 $N(0,\Delta t/2)$) 라 쓰면 중점값은 $B_{t_i}+\delta_i^{(1)}$ 이므로
$$M_n=\sum_i(B_{t_i}+\delta_i^{(1)})(\delta_i^{(1)}+\delta_i^{(2)})=L_n+\sum_i(\delta_i^{(1)})^2+\sum_i\delta_i^{(1)}\delta_i^{(2)} .$$
$\sum(\delta^{(1)})^2$ 는 평균 $n\cdot\Delta t/2=T/2$, 분산 $n\cdot2(\Delta t/2)^2=T^2/(2n)\to0$; 교차합은 평균 0, 분산 $n(\Delta t/2)^2=T^2/(4n)\to0$. 따라서 $M_n\to\tfrac{B_T^2-T}2+\tfrac T2=\tfrac{B_T^2}2$ ($L^2$).

오차는 $QV_n=\sum(\delta^{(1)})^2+\sum(\delta^{(2)})^2+2\sum\delta^{(1)}\delta^{(2)}$ 를 대입하면
$M_n-\tfrac{B_T^2}2=-\tfrac12QV_n+\sum(\delta^{(1)})^2+\sum\delta^{(1)}\delta^{(2)}=\tfrac12\Big[\sum_i(\delta_i^{(1)})^2-\sum_i(\delta_i^{(2)})^2\Big]$
이고 두 합은 독립이므로 분산은 $\tfrac14\cdot2\cdot\tfrac{T^2}{2n}=\tfrac{T^2}{4n}$. ∎

</details>

### 2.4 정리 3 — Itô 적분의 마팅게일 성질 (Øksendal §3.2)

$H$ 가 적합이고 $E\int_0^TH^2dt<\infty$ 이면 $M_t=\int_0^tH\,dB$ 는 $\mathcal F_t$-마팅게일이고 연속 버전을 가진다.

가정이 왜 필요한가:
- **적합성**: $\xi_j$ 가 $\mathcal F_{t_j}$-가측이어야 $E[\xi_j\Delta B_j\mid\mathcal F_{t_j}]=\xi_j\,E[\Delta B_j\mid\mathcal F_{t_j}]=0$ 이 되고 (05a의 'taking out what is known' — 단, $\mathcal F_s$ 가 아니라 **$\mathcal F_{t_j}$-조건부 기대**에서 꺼낸다), 그다음 tower property 로 $E[\xi_j\Delta B_j\mid\mathcal F_s]=E\big[E[\xi_j\Delta B_j\mid\mathcal F_{t_j}]\mid\mathcal F_s\big]=0$ ($t_j\ge s$). $t_j>s$ 이면 $\xi_j$ 는 $\mathcal F_s$-가측이 아니므로 $E[\cdot\mid\mathcal F_s]$ 밖으로 바로 꺼낼 수 없다.
- **$L^2$ 조건**: 마팅게일 성질이 $L^2$ 극한에서 보존되려면 조건부 기대의 $L^2$ 연속성($\|E[X\mid\mathcal G]\|_2\le\|X\|_2$)이 필요하다.

*증명 스케치.* 단순 과정: $s\in[t_k,t_{k+1})$ 라 하면 $M_T-M_s=\xi_k(B_{t_{k+1}}-B_s)+\sum_{j>k}\xi_j\Delta B_j$. 첫 항은 $\xi_k$ 가 $\mathcal F_s$-가측이고 $B_{t_{k+1}}-B_s\perp\mathcal F_s$ 라 조건부 기대가 0; 나머지는 tower property 로 $E[\xi_j\Delta B_j\mid\mathcal F_s]=E[\xi_jE[\Delta B_j\mid\mathcal F_{t_j}]\mid\mathcal F_s]=0$. 일반 $H$ 는 $L^2$ 극한. 연속 버전은 Doob 부등식(05e)으로 근사열이 균등하게 수렴함을 보인다 — Øksendal §3.2 Thm 3.2.5 를 인용한다. $\square$

### 2.5 직관

리만–스틸체스 적분은 '피적분함수가 구간 안에서 거의 안 변한다'는 가정 위에 서 있다. 브라운 운동은 총변동이 무한이라 $\sum\Delta B_i^2$ 가 0이 아니라 $T$ 로 남고, 이 $T$ 가 표본점을 왼쪽에서 오른쪽으로 옮길 때 그대로 극한값의 차이($T$)가 된다. Itô는 '왼쪽 끝점 = 지금까지의 정보만으로 베팅'을 택한다. 그러면 $\xi_i$ 와 $\Delta B_i$ 가 독립이 되어 (i) 평균 0 (공정 게임), (ii) 분산이 $\sum E[\xi_i^2]\Delta t$ 로 깔끔히 더해진다(등거리성). 즉 Itô 적분은 '도박꾼이 매 순간 판돈 $H_t$ 를 걸고 공정한 동전 $dB_t$ 를 받는 누적 손익'이며 05b의 마팅게일 변환 $\sum H_{n-1}\Delta M_n$ 의 연속시간판이다.

### 2.6 함정 (traps)

1. "리만합의 극한은 표본점에 무관하다"는 매끄러운 피적분함수·유계변동 적분자에만 해당한다. BM 에서는 $L_n,M_n,R_n$ 의 극한이 $T/2$ 씩 다르다.
2. $E[\int H\,dB]=0$ 은 적합성 없이는 거짓: 오른쪽 끝점 합 $R_n$ 의 평균은 $T$ 다.
3. 등거리성은 분산만 준다, 분포는 아니다: $\int_0^Tt\,dB$ 는 가우스지만 $\int_0^TB\,dB=(B_T^2-T)/2$ 는 $(\chi^2_1-1)T/2$ 로 비대칭이다.
4. Itô 적분은 경로별(pathwise) 리만–스틸체스 적분이 아니라 $L^2$ 극한이다. 그러나 왼쪽 끝점 합은 확률적으로 수렴하므로 시뮬레이션은 왼쪽 합으로 한다.
5. 이산 등거리성 $E[(\sum\xi_i\Delta B_i)^2]=\sum E[\xi_i^2]\Delta t$ 는 어떤 $n$ 에서도 **정확**하다. 연속 값 $E\int H^2dt$ 와의 차이는 리만합 오차 $O(\Delta t)$ 뿐이다.

쓰기 과제: 다음 셀에 'Itô 등거리성이 성립하려면 왜 판돈 $H$ 를 구간의 왼쪽 끝에서 정해야 하는가'를 도박꾼 비유로 3문장으로 쓰고, 오른쪽 끝에서 정하면 평균이 얼마가 되는지 적어 보세요.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 아래 값을 먼저 적어 보세요. 손계산 힌트: $E[B_t^2]=t$, $E[\cos 2B_t]=e^{-2t}$, $\mathrm{Var}(QV_n)=2T^2/n$. 모두 $T=1$.

In [ ]:
predictions = {
    # 왼쪽 끝점 리만합 L_n = sum B_{t_i} dB_i 의 평균 E[L_n] 은? (n 에 무관)
    "mean_ito": None,
    # 중점 리만합 M_n 의 평균 (= E[B_1^2/2]) 은?
    "mean_strat": None,
    # E[(∫_0^1 B dB)^2] 은? (등거리성, 또는 (B_1^2-1)/2 의 분산)
    "var_ito": None,
    # E[(∫_0^1 t dB_t)^2] 은?
    "iso_t": None,
    # E[(∫_0^1 cos(B_t) dB_t)^2] 은? (힌트: E[cos^2 B_t] = (1 + e^{-2t})/2)
    "iso_cosB": None,
    # n=500 일 때 왼쪽 합의 오차 L_n - (B_1^2-1)/2 의 RMS 는?
    "rms_left_500": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 경로 생성과 세 리만합 (Fig 1)

세밀 격자($n=1000$)에서 경로를 만들고, `coarsen` 으로 $n=500$ 격자를 얻는다. 세밀 격자의 홀수 인덱스가 정확히 $n=500$ 구간의 중점이므로 중점 합도 **같은 경로**에서 계산할 수 있다.

In [ ]:
t, B = brownian_paths(N_PATHS, N_FINE, T, rng)      # (N_PATHS, 1001)
tc, Bc = coarsen(t, B, 2)                            # n = 500 격자
N_SUM = tc.size - 1
dBc = np.diff(Bc, axis=1)
Bmid = B[:, 1::2]                                    # 홀수 인덱스 = 각 구간의 중점

L = (Bc[:, :-1] * dBc).sum(1)      # 왼쪽 끝점 (Itô)
M = (Bmid * dBc).sum(1)            # 중점 (Stratonovich)
R = (Bc[:, 1:] * dBc).sum(1)       # 오른쪽 끝점 (미래 엿보기)

BT = B[:, -1]
ito, strat, right = (BT**2 - T) / 2, BT**2 / 2, (BT**2 + T) / 2
print(f"n_sum={N_SUM}; one path: L={L[0]:.4f} vs {ito[0]:.4f}, "
      f"M={M[0]:.4f} vs {strat[0]:.4f}, R={R[0]:.4f} vs {right[0]:.4f}")

In [ ]:
k = 0   # 한 경로
cumL, cumM, cumR = (np.cumsum(Bc[k, :-1] * dBc[k]), np.cumsum(Bmid[k] * dBc[k]),
                    np.cumsum(Bc[k, 1:] * dBc[k]))
tt = tc[1:]
closed = [(Bc[k, 1:]**2 - tt) / 2, Bc[k, 1:]**2 / 2, (Bc[k, 1:]**2 + tt) / 2]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, lo in zip(axes, [0.0, 0.8]):
    for cum, cf, lab, c in zip([cumL, cumM, cumR], closed, ["left", "midpoint", "right"], ["C0", "C1", "C2"]):
        m = tt >= lo
        ax.plot(tt[m], cum[m], color=c, lw=1.2, label=f"{lab} sum")
        ax.plot(tt[m], cf[m], color=c, ls="--", lw=1.0, alpha=0.8)
    ax.set_xlabel("t")
axes[0].set_title("Running Riemann sums for ∫B dB on one path (n=500)\ndashed: (B²-t)/2, B²/2, (B²+t)/2")
axes[1].set_title("zoom: last 0.2 of time")
axes[0].legend(); plt.tight_layout()

세 누적합은 처음부터 서로 갈라져 각각 자기 닫힌 형식을 따라간다. 격차 $t/2$ 는 시간에 비례해 커진다 — 지금까지 쌓인 이차변동의 절반이다.

### 4.2 세 합의 분포 (Fig 2)

세 **극한**은 모두 $B_T^2/2$ 의 평행이동이므로 분산은 같고($\mathrm{Var}(B_T^2)/4=T^2/2$) 평균만 $T/2$ 씩 다르다. 유한 $n$ 에서는 $R_n-L_n=QV_n$ 이 확률변수라 세 합의 분산이 $O(1/n)$ 만큼 다르다 ($n=500$ 에서는 거의 같다). 왼쪽 합에는 $(\chi^2_1-1)T/2$ 의 밀도를 겹친다.

In [ ]:
def chi2m1_half_pdf(x):
    """(χ²₁ − 1)/2 의 밀도: y = 2x + 1 ~ χ²₁ 이므로 pdf(x) = 2·chi2(1).pdf(2x+1)."""
    x = np.asarray(x, dtype=float)
    return np.where(x > -0.5, 2 * stats.chi2(1).pdf(2 * x + 1), 0.0)

bins = np.linspace(-0.6, 4.5, 70)
xs = np.linspace(-0.49, 4.5, 400)
fig, ax = plt.subplots()
plot_hist_vs_pdf(L, xs, chi2m1_half_pdf(xs), ax=ax, bins=bins, label_pdf="density of (chi2_1 - 1)/2")
ax.hist(M, bins=bins, density=True, alpha=0.35, color="C2", label="midpoint sum M_n")
ax.hist(R, bins=bins, density=True, alpha=0.35, color="C3", label="right sum R_n")
ax.set_ylim(0, 2.0); ax.set_xlabel("value of the Riemann sum"); ax.set_ylabel("density")
ax.set_title("Left / midpoint / right sums (n=500): same shape, shifted by T/2 each")
h, lab = ax.get_legend_handles_labels()
ax.legend(h, ["left sum L_n" if l == "simulation" else l for l in lab])

est_L, est_M, est_R = mc_estimate(L), mc_estimate(M), mc_estimate(R)
est_L2 = mc_estimate(L**2)
for nm, e in [("E[L]", est_L), ("E[M]", est_M), ("E[R]", est_R), ("E[L^2]", est_L2)]:
    print(f"{nm:7s} = {e}")
print(f"Var(L), Var(M), Var(R) = {L.var():.3f}, {M.var():.3f}, {R.var():.3f}  (exact T^2/2 = {T**2/2})")

분포는 $-T/2$ 에서 급격히 잘리고 오른쪽 꼬리가 긴 비대칭 — 등거리성이 분산 $T^2/2$ 를 주지만 "가우스"라고는 말하지 않는다(함정 3).

### 4.3 격자 세분화에 따른 수렴 (Fig 3)

같은 1000-격자 경로를 `coarsen` 으로 성기게 보며 $n\in\{10,\dots,1000\}$ 에서 왼쪽 합의 RMS 오차를 잰다. 극한 `ito` 는 $B_T$ 만 쓰므로 모든 해상도에서 같다. 중점 합은 factor 가 짝수여야 세밀 격자 위에 중점이 있다.

In [ ]:
factors = [100, 50, 20, 10, 5, 2, 1]
ns, rms_left, rms_mid = [], [], []
for f in factors:
    tk, Bk = coarsen(t, B, f)
    n_k = tk.size - 1
    dBk = np.diff(Bk, axis=1)
    Lk = (Bk[:, :-1] * dBk).sum(1)
    ns.append(n_k)
    rms_left.append(np.sqrt(((Lk - ito) ** 2).mean()))
    if f % 2 == 0:                                  # 중점 = 세밀 격자의 f/2 오프셋
        Mk = (B[:, f // 2::f] * dBk).sum(1)
        rms_mid.append((n_k, np.sqrt(((Mk - strat) ** 2).mean())))
ns = np.array(ns); rms_left = np.array(rms_left)
n_mid, r_mid = map(np.array, zip(*rms_mid))
slope_left = np.polyfit(np.log(ns), np.log(rms_left), 1)[0]
slope_mid = np.polyfit(np.log(n_mid), np.log(r_mid), 1)[0]
print("n        :", ns)
print("RMS left :", np.round(rms_left, 4))
print("T/sqrt(2n):", np.round(T / np.sqrt(2 * ns), 4))
print(f"log-log slopes: left {slope_left:.3f}, midpoint {slope_mid:.3f} (theory -1/2)")

In [ ]:
fig, ax = plt.subplots()
ax.loglog(ns, rms_left, "o-", label="left sum: RMS(L_n - (B_T²-T)/2)")
ax.loglog(ns, T / np.sqrt(2 * ns), "k--", lw=1, label="T / sqrt(2n)")
ax.loglog(n_mid, r_mid, "s-", color="C1", label="midpoint sum: RMS(M_n - B_T²/2)")
ax.loglog(n_mid, T / (2 * np.sqrt(n_mid)), "k:", lw=1, label="T / (2 sqrt n)")
ax.set_xlabel("n (number of subintervals)"); ax.set_ylabel("RMS error")
ax.set_title(f"RMS error of Riemann sums vs n (slopes {slope_left:.2f}, {slope_mid:.2f})")
ax.legend()
est_err_L = mc_estimate((L - ito) ** 2)      # n = 500
est_err_M = mc_estimate((M - strat) ** 2)
print(f"n=500: E[(L-ito)^2] = {est_err_L:.6f}  exact T^2/(2n) = {T**2/(2*N_SUM):.6f}")
print(f"n=500: E[(M-strat)^2] = {est_err_M:.6f}  exact T^2/(4n) = {T**2/(4*N_SUM):.6f}")

기울기 $-1/2$: 오차는 $\sqrt{\Delta t}$ 차수로 줄어든다. 중점 합은 상수가 $1/\sqrt2$ 배 작을 뿐 차수는 같다. $L^2$ 수렴이 실제로 이 속도로 일어나는 것을 **같은 $\omega$** 에서 확인한 것이다.

### 4.4 Itô 등거리성 표 (Fig 4)

별도의 경로 집합($n=400$)에서 네 피적분함수 $H\in\{1,\ t,\ B_t,\ \cos B_t\}$ 에 대해 $I(H)=\sum H_{t_i}\Delta B_i$ 를 만들고 $E[I]=0$ 과 $E[I^2]=E\int H^2dt$ 를 확인한다. 연속 닫힌 형식 $\int_0^1E[H_t^2]dt$ 와 이산 정확값 $\sum_iE[H_{t_i}^2]\Delta t$ (함정 5) 를 나란히 둔다.

In [ ]:
N_STEPS2 = 400
t2, B2 = brownian_paths(N_ISO, N_STEPS2, T, rng)
dB2 = np.diff(B2, axis=1)
Bl, tl, dt2 = B2[:, :-1], t2[:-1], T / N_STEPS2          # 왼쪽 끝점의 값과 시각

integrands = {"1": np.ones_like(Bl), "t": np.broadcast_to(tl, Bl.shape),
              "B_t": Bl, "cos B_t": np.cos(Bl)}
I = {nm: (H * dB2).sum(1) for nm, H in integrands.items()}
I1, It, IB, Icos = I["1"], I["t"], I["B_t"], I["cos B_t"]

# 연속 닫힌 형식 ∫_0^1 E[H_t^2] dt  와  이산 정확값 Σ E[H_{t_i}^2] Δt
iso_cont = {"1": 1.0, "t": 1 / 3, "B_t": T**2 / 2, "cos B_t": 0.5 + (1 - np.exp(-2 * T)) / 4}
EH2 = {"1": np.ones_like(tl), "t": tl**2, "B_t": tl, "cos B_t": (1 + np.exp(-2 * tl)) / 2}
iso_disc = {nm: v.sum() * dt2 for nm, v in EH2.items()}

est_mean = {nm: mc_estimate(x) for nm, x in I.items()}
est_iso = {nm: mc_estimate(x**2) for nm, x in I.items()}
print(f"{'H':8s} {'E[I]':>22s}   {'E[I^2] (MC)':>22s}   cont.    disc.    bias/SE")
for nm in I:
    print(f"{nm:8s} {est_mean[nm].mean:8.4f} ± {est_mean[nm].se:.4f}   "
          f"{est_iso[nm].mean:8.4f} ± {est_iso[nm].se:.4f}   {iso_cont[nm]:.5f}  "
          f"{iso_disc[nm]:.5f}  {(iso_cont[nm]-iso_disc[nm])/est_iso[nm].se:+.2f}")

In [ ]:
names = list(I)
x = np.arange(len(names))
means = [est_iso[nm].mean for nm in names]
errs = [1.96 * est_iso[nm].se for nm in names]
fig, ax = plt.subplots()
ax.bar(x, means, yerr=errs, capsize=5, alpha=0.6, label="MC E[I(H)²] ± 1.96 SE")
ax.plot(x, [iso_cont[nm] for nm in names], "k_", ms=25, mew=2, label="∫ E[H²] dt (isometry)")
ax.set_xticks(x); ax.set_xticklabels([f"H = {nm}" for nm in names])
ax.set_ylabel("E[(∫H dB)²]")
ax.set_title("Itô isometry: MC vs closed form, four integrands (T=1)")
ax.legend()

리만합 편향(연속값 − 이산값)은 SE 의 일부에 불과하다: 등거리성은 이산에서 정확하고, 연속 극한과의 차이는 $O(\Delta t)$ 다.

### 4.5 마팅게일 성질 (Fig 5)

4.1 의 세밀 경로(1000 격자)에서 $M_s=\int_0^sB\,dB$ ($s=0.5$) 와 $M_T=\int_0^1B\,dB$ 를 만든다. 직교 증분 $E[M_s(M_T-M_s)]=0$, 따라서 $E[M_sM_T]=E[M_s^2]=\int_0^st\,dt=s^2/2$. 그리고 $E[M_T\mid M_s]=M_s$ 를 분위 구간별 조건부 평균으로 본다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def binned_conditional_mean(x, y, n_bins=10):
    """x 의 분위 구간마다 (x 평균, y 평균, y 의 SE, 개수) 를 반환한다."""
    edges = np.quantile(x, np.linspace(0, 1, n_bins + 1))
    idx = np.clip(np.searchsorted(edges, x, side="right") - 1, 0, n_bins - 1)
    out = []
    for b in range(n_bins):
        m = idx == b
        out.append((x[m].mean(), y[m].mean(), y[m].std(ddof=1) / np.sqrt(m.sum()), m.sum()))
    return np.array(out)

s_idx = N_FINE // 2                      # s = 0.5
s = t[s_idx]
dB = np.diff(B, axis=1)
Ms = (B[:, :s_idx] * dB[:, :s_idx]).sum(1)
MT = (B[:, :-1] * dB).sum(1)
est_orth = mc_estimate(Ms * (MT - Ms))
est_MsMT = mc_estimate(Ms * MT)
ex_MsMT_cont = s**2 / 2
ex_MsMT_disc = (t[:s_idx]).sum() * (T / N_FINE)      # Σ_{t_i<s} t_i Δt
print(f"E[M_s (M_T - M_s)] = {est_orth}   (exact 0)")
print(f"E[M_s M_T]         = {est_MsMT}   (cont. s^2/2 = {ex_MsMT_cont:.5f}, disc. {ex_MsMT_disc:.5f})")

In [ ]:
cm = binned_conditional_mean(Ms, MT, n_bins=10)
fig, ax = plt.subplots()
ax.errorbar(cm[:, 0], cm[:, 1], yerr=2 * cm[:, 2], fmt="o", capsize=4, label="E[M_1 | M_0.5 in bin] ± 2 SE")
lim = [cm[:, 0].min() - 0.1, cm[:, 0].max() + 0.1]
ax.plot(lim, lim, "k--", lw=1, label="y = x")
ax.set_xlabel("M_0.5 (bin mean)"); ax.set_ylabel("mean of M_1 in bin")
ax.set_title("Martingale check: E[M_1 | M_0.5] = M_0.5 (10 quantile bins)")
ax.legend()

점들이 대각선 위에 놓인다: 시각 $0.5$ 까지의 누적 손익이 얼마였든, 이후의 기대 손익은 0 이다. 이것이 $E[M_T\mid\mathcal F_s]=M_s$ 의 그림이다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드에서 계산한다: 정리 2 의 평균·평균제곱오차 공식, 정리 1 의 $\int_0^1E[H_t^2]dt$, 정리 3 의 $s^2/2$. 표의 '닫힌 형식' 은 연속 극한값이지만, 리만합 편향이 있는 세 행($E[L^2]$, $H=t$, $E[M_sM_T]$)의 `assert_close` 는 이산 정확값(함정 5)을 쓴다.

In [ ]:
exact = {
    "mean_ito": 0.0, "mean_strat": T / 2, "mean_right": T,
    "var_ito": T**2 / 2,
    "mse_left": T**2 / (2 * N_SUM), "mse_mid": T**2 / (4 * N_SUM),
    "iso_1": iso_cont["1"], "iso_t": iso_cont["t"], "iso_B": iso_cont["B_t"], "iso_cosB": iso_cont["cos B_t"],
    "orth": 0.0, "MsMT": s**2 / 2,
}
# 이산 정확값 (함정 5): 표에는 연속 닫힌 형식을 두고, assert 는 추정량의 참값인 이 값으로 한다
exact_disc = {"var_ito": T**2 * (N_SUM - 1) / (2 * N_SUM), "iso_t": iso_disc["t"], "MsMT": ex_MsMT_disc}
p = predictions
rows = [
    {"name": "E[left sum L_500] (Itô)", "predicted": p["mean_ito"], "estimate": est_L, "exact": exact["mean_ito"]},
    {"name": "E[midpoint sum M_500] (Stratonovich)", "predicted": p["mean_strat"], "estimate": est_M, "exact": exact["mean_strat"]},
    {"name": "E[right sum R_500]", "predicted": None, "estimate": est_R, "exact": exact["mean_right"]},
    {"name": "E[L_500^2] (isometry, H=B)", "predicted": p["var_ito"], "estimate": est_L2, "exact": exact["var_ito"]},
    {"name": "E[(L_500 - (B_1^2-1)/2)^2]  (내 예측 = RMS²)", "predicted": None if p["rms_left_500"] is None else p["rms_left_500"]**2,
     "estimate": est_err_L, "exact": exact["mse_left"]},
    {"name": "E[(M_500 - B_1^2/2)^2]", "predicted": None, "estimate": est_err_M, "exact": exact["mse_mid"]},
    {"name": "isometry H=1: E[B_1^2]", "predicted": None, "estimate": est_iso["1"], "exact": exact["iso_1"]},
    {"name": "isometry H=t", "predicted": p["iso_t"], "estimate": est_iso["t"], "exact": exact["iso_t"]},
    {"name": "isometry H=B_t (n=400 set)", "predicted": None, "estimate": est_iso["B_t"], "exact": exact["iso_B"]},
    {"name": "isometry H=cos(B_t)", "predicted": p["iso_cosB"], "estimate": est_iso["cos B_t"], "exact": exact["iso_cosB"]},
    {"name": "E[M_0.5 (M_1 - M_0.5)] (orthogonal incr.)", "predicted": None, "estimate": est_orth, "exact": exact["orth"]},
    {"name": "E[M_0.5 M_1] = E[M_0.5^2]", "predicted": None, "estimate": est_MsMT, "exact": exact["MsMT"]},
]
Markdown(compare_table(rows))

각 행이 어긋난다면 무엇이 의심되는가:
- **E[L], E[M], E[R]**: 셋 중 하나만 틀리면 표본점 인덱싱(`Bc[:, :-1]`, `B[:, 1::2]`, `Bc[:, 1:]`) 이 밀렸다. 셋이 같은 값이면 이차변동이 사라진 것 — BM 이 아니다.
- **E[L_500^2]**: 등거리성 자체가 의심 대상. 단, 표의 닫힌 형식은 연속값 $1/2$ 이고 이산 정확값은 $T^2(n-1)/(2n)$ 이라 그 차이 $1/(2n)$ 는 SE 안에 묻힌다 (assert 는 이산값으로).
- **MSE 행 두 개**: $\mathrm{Var}(QV_n)=2T^2/n$ (06b) 이 틀렸거나, 중점 오차 공식의 $\tfrac14\cdot2$ 인자를 놓쳤다. SE 가 아주 작아 가장 민감한 검사다.
- **isometry 네 행**: $E[H_t^2]$ 의 닫힌 형식(특히 $E[\cos^2B_t]=(1+e^{-2t})/2$) 이 틀렸거나 $H$ 를 오른쪽 끝점에서 평가했다.
- **직교 증분·E[M_s M_T]**: `Ms` 가 `dB[:, :500]` 이 아니라 다른 구간을 더했거나, 마팅게일 성질이 깨졌다(적합성 위반).

In [ ]:
assert_close(est_L, exact["mean_ito"], k=4, name="E[L]")
assert_close(est_M, exact["mean_strat"], k=4, name="E[M]")
assert_close(est_R, exact["mean_right"], k=4, name="E[R]")
assert_close(est_L2, exact_disc["var_ito"], k=4, name="E[L^2]")            # 이산 정확값 (함정 5)
assert_close(est_err_L, exact["mse_left"], k=4, name="msq err left n=500")
assert_close(est_err_M, exact["mse_mid"], k=4, name="msq err mid n=500")
assert_close(est_iso["1"], exact["iso_1"], k=4, name="iso H=1")
assert_close(est_iso["t"], exact_disc["iso_t"], k=4, name="iso H=t")       # 이산 정확값 (함정 5)
assert_close(est_iso["B_t"], exact["iso_B"], k=4, name="iso H=B")
assert_close(est_iso["cos B_t"], exact["iso_cosB"], k=4, name="iso H=cos B")
assert_close(est_orth, exact["orth"], k=4, name="orthogonal increments")
assert_close(est_MsMT, exact_disc["MsMT"], k=4, name="E[Ms MT]")          # 이산 정확값 (함정 5)
print("all assert_close passed (k=4)")

## 6. 연습문제

### E1 계산

$T=2$ 로 바꾸면 세 극한 $(B_T^2-T)/2$, $B_T^2/2$, $(B_T^2+T)/2$ 의 평균과 분산은 각각 얼마인가? 또 $L_n$ 의 평균제곱오차 $E[(L_n-\tfrac{B_T^2-T}2)^2]$ 를 $n=200$ 에서 손으로 구하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$B_T^2\sim T\chi^2_1$ 이므로 $E[B_T^2]=T=2$, $\mathrm{Var}(B_T^2)=2T^2=8$.

- 평균: $\tfrac12(T-T)=0$, $\tfrac12T=1$, $\tfrac12(T+T)=2$.
- 분산: 세 극한 모두 $B_T^2/2$ 의 평행이동이므로 $\mathrm{Var}(B_T^2)/4=T^2/2=2$.
- 오차 $L_n-\tfrac{B_T^2-T}2=\tfrac12(T-QV_n)$, $\mathrm{Var}(QV_n)=2T^2/n=8/200$ → MSE $=\tfrac14\cdot\tfrac{8}{200}=T^2/(2n)=0.01$, RMS $=0.1$.

검산 코드 (다른 `T` 에서 4.1 을 반복):

```python
T2, n2 = 2.0, 200
tt2, BB2 = brownian_paths(N_PATHS, n2, T2, rng)
dd = np.diff(BB2, axis=1); BT2 = BB2[:, -1]
L2_ = (BB2[:, :-1] * dd).sum(1)
print(mc_estimate(L2_), mc_estimate(L2_**2), "exact var", T2**2 / 2 * (n2 - 1) / n2)
print(mc_estimate((L2_ - (BT2**2 - T2) / 2) ** 2), "exact MSE", T2**2 / (2 * n2))
```

</details>

### E2 유도 후 시뮬레이션 검증

등거리성으로 $E[(\int_0^TB_t^2\,dB_t)^2]$ 을 유도하고 ($E[B_t^4]=3t^2$ 사용), $T=1$ 에서 4.4 의 경로 집합(`Bl`, `dB2`)으로 검증하라. 덤으로 편광(polarization) 항등식 $E[\int H\,dB\int G\,dB]=E\int HG\,dt$ 를 써서 $E[(\int_0^1t\,dB)(\int_0^1B\,dB)]$ 를 구하고 검증하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

등거리성: $E[(\int_0^TB^2dB)^2]=\int_0^TE[B_t^4]dt=\int_0^T3t^2dt=T^3$, $T=1$ 이면 $1$. 이산 정확값은 $3\sum_it_i^2\Delta t=0.99625$ ($n=400$).

편광은 등거리성을 $H+G$ 와 $H-G$ 에 적용해 빼면 나온다: $E[\int H\,dB\int G\,dB]=\tfrac14\big(E\int(H+G)^2dt-E\int(H-G)^2dt\big)=E\int HG\,dt$. $H=t$, $G=B_t$ 이면 $\int_0^1t\,E[B_t]\,dt=0$.

```python
IB2 = (Bl**2 * dB2).sum(1)
est_B2 = mc_estimate(IB2**2)
print(est_B2, "exact T^3 =", T**3, "discrete", 3 * (tl**2).sum() * dt2)   # relSE ≈ 3%, 4 SE 이내
assert_close(est_B2, T**3, k=4, name="iso H=B^2")
est_pol = mc_estimate(It * IB)
print(est_pol, "exact 0")
assert_close(est_pol, 0.0, k=4, name="polarization t, B")
```

$B_t^2$ 피적분함수는 꼬리가 무거워($\mathrm{Var}(I^2)$ 에 $E[I^4]$, 즉 $B_t$ 의 8차 모멘트 $E[B_t^8]=105t^4$ 가 들어감) SE 가 다른 행보다 크다. 이것이 본문 표에서 $e^{B_t}$ 같은 피적분함수 대신 $\cos B_t$ 를 고른 이유다.

</details>

### E3 가정을 깨보기

적합성을 깨보자: 피적분함수를 오른쪽 끝점 $B_{t_{i+1}}$ (미래를 엿봄) 으로 두면 $R_n$ 이 된다. $E[R_n]$ 과 $E[R_n^2]$ 를 계산하고 등거리성이 예측하는 $E\int_0^1B^2dt=1/2$ 와 비교하라. 시뮬레이션으로 $n=500$ 에서 확인하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$R_n=L_n+QV_n$ 이므로 $E[R_n]=0+T=1\ne0$: 공정 게임이 아니다.

제곱: $E[R_n^2]=E[L_n^2]+2E[L_nQV_n]+E[QV_n^2]$. 여기서 $L_n=\tfrac12(B_T^2-QV_n)$ 이고, $B_T^2=\sum_i\Delta B_i^2+2\sum_{i<j}\Delta B_i\Delta B_j$ 에서 교차항은 $QV_n$ 과 곱해도 평균 0 이므로 $E[B_T^2QV_n]=E[QV_n^2]=T^2+2T^2/n$. 따라서 $E[L_nQV_n]=\tfrac12(E[B_T^2QV_n]-E[QV_n^2])=0$ 이고

$$E[R_n^2]=\frac{T^2(n-1)}{2n}+T^2+\frac{2T^2}n\ \xrightarrow{n=500}\ 0.499+1+0.004=1.503,\qquad\lim_n=\tfrac32T^2=1.5\ne\tfrac12 .$$

등거리성이 예측하는 $1/2$ 의 세 배다. 미래 정보를 쓰면 평균 0 도, 분산 공식도 무너진다.

```python
est_R2 = mc_estimate(R**2)
ex_R2 = T**2 * (N_SUM - 1) / (2 * N_SUM) + T**2 + 2 * T**2 / N_SUM
print(est_R, "exact E[R] =", T)
print(est_R2, "exact E[R^2] =", ex_R2, "; isometry would say", T**2 / 2)
assert_close(est_R2, ex_R2, k=4, name="E[R^2] (non-adapted)")
```

</details>

## 7. 정리

1. BM 에 대한 리만합은 표본점에 따라 극한이 다르다: 왼쪽 $(B_T^2-T)/2$ (Itô), 중점 $B_T^2/2$ (Stratonovich), 오른쪽 $(B_T^2+T)/2$ — 차이는 정확히 이차변동 $T$ 의 절반씩.
2. Itô 적분은 왼쪽 끝점(적합) 합의 $L^2$ 극한이며, 적합성 덕분에 $E[\int H\,dB]=0$ 이고 마팅게일이다 ($E[M_1\mid M_{0.5}]=M_{0.5}$ 를 그림으로 확인).
3. Itô 등거리성 $E[(\int H\,dB)^2]=E\int H^2dt$ 는 분산을 계산해 주고 $L^2$ 극한의 존재를 보장한다; 이산 버전은 어떤 $n$ 에서도 정확하다 ($H=1,t,B_t,\cos B_t$ 모두 4 SE 이내).
4. 왼쪽 합의 오차 RMS 는 $T/\sqrt{2n}$ (기울기 $-1/2$), 중점 합은 $T/(2\sqrt n)$ — 같은 경로를 `coarsen` 해서 측정했다.
5. 적합성을 깨면(오른쪽 끝점) 평균이 $T$, 제곱평균이 $3T^2/2$ 로 튀어 등거리성이 무너진다.

**Trap 카드**
- Q: $\int_0^TB\,dB=B_T^2/2$ 인가?
- A: 아니다. Itô(왼쪽 끝점)에서는 $(B_T^2-T)/2$ 이고, $B_T^2/2$ 는 중점(Stratonovich) 극한이다. 차이 $T/2$ 는 이차변동에서 온다: $\sum\Delta B_i^2\to T\ne0$.

**다음 노트북: 07b** — $(B_T^2-T)/2$ 는 사실 $f(x)=x^2$ 에 대한 Itô 공식 $f(B_T)=f(0)+\int f'(B)\,dB+\tfrac12\int f''(B)\,dt$ 의 특수한 경우다. 07b 에서 이 공식을 일반 $f$ 로 확장하고, 드리프트 항 $\tfrac12f''$ 이 어디서 오는지($dB^2=dt$) 본다.

log/progress.md 한 줄 템플릿:
`- [ ] 07a 이토 적분과 등거리성 — YYYY-MM-DD, 예측 6개 중 __개 적중, 막힌 곳: ____, 다시 볼 것: 정리 2 중점 오차 T²/(4n) 유도`